# Project – Personalizing the Chatbot with System Prompts

## Overview

## Assemble and Launch Personalized Chatbot

In [27]:
## === Imports ===
import json
import random

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

## === OpenAI Setup ===
load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"

## === Simulation constants ===
# MAX_TURNS                    -> conversation "stalls" after this many employee turns
# MIN_CATEGORIES_FOR_DIAGNOSIS -> must ask about at least 2 areas before a diagnosis/offer counts
# MAX_PARTS_DISCOUNT           -> repairable: parts only, never labor
# MAX_NEW_PHONE_DISCOUNT       -> not repairable: new phone, trade-in required
# ACCEPT_THRESHOLD             -> satisfaction needed for the customer to accept a fair offer
MAX_VIOLATIONS = 3
MAX_TURNS = 25
MIN_CATEGORIES_FOR_DIAGNOSIS = 2
CATEGORIES = ["environmental", "physical", "power", "timeline"]
MAX_PARTS_DISCOUNT = 40
MAX_NEW_PHONE_DISCOUNT = 30
ACCEPT_THRESHOLD = 5.0

# How much each behaviour moves satisfaction 
DELTAS = {
    "polite": 0.5,
    "empathy": 0.75,
    "new_category": 0.5,
    "deferred_discount": 0.5,
    "correct_diagnosis": 1.5,
    "valid_offer": 2.5,
    "accusatory": -1.5,
    "premature_offer": -0.5,
    "wrong_diagnosis": -1.0,
    "violation": -2.0,
}

## ===== Scenarios =============================================================
SCENARIOS = {
    "liquid": {
        "label": "Liquid damage",
        "cause": "Liquid damage, corroded logic board",
        "repairable": False,
        "clues": {
            "environmental": "Two days ago the phone was in your pocket at a pool party and went into the water for a few seconds when you got pushed in. You're a little embarrassed about it.",
            "physical": "You haven't dropped it. No cracks. It was in a thin case.",
            "power": "It worked for a few hours after the pool, got warm, then died. There's some greenish crust in the charging port. A different cable didn't help.",
            "timeline": "It died two days ago. It's about 18 months old, bought new, and nobody has opened it.",
        },
    },
    "heat": {
        "label": "Heat / swollen battery",
        "cause": "Heat damage, battery swelling",
        "repairable": True,
        "clues": {
            "environmental": "Last week you left it on your car dashboard all afternoon during a heat wave.",
            "physical": "No drops, but the back looks slightly bulged and the case doesn't snap on properly anymore.",
            "power": "After the car it got very hot, the battery drained fast for a day or two, then it shut off and won't turn on.",
            "timeline": "Started about five days ago. The phone is two years old, bought new, never repaired.",
        },
    },
    "impact": {
        "label": "Drop / impact",
        "cause": "Physical impact damage (display/connector)",
        "repairable": True,
        "clues": {
            "environmental": "No water, heat or cold that you can think of.",
            "physical": "Yesterday you dropped it on concrete from about chest height. No case. The screen has a crack in one corner.",
            "power": "Since the drop the screen stays black, but you think it still vibrates when someone calls.",
            "timeline": "It happened yesterday. The phone is a year old, bought new, nobody has opened it.",
        },
    },
    "battery": {
        "label": "Battery end-of-life",
        "cause": "Battery end-of-life",
        "repairable": True,
        "clues": {
            "environmental": "Nothing unusual — no water, heat or cold.",
            "physical": "No drops or cracks that you know of.",
            "power": "Over the past few weeks the battery drained faster and faster and it shut off randomly around 30%. Now it won't turn on even after charging overnight. You tried another cable.",
            "timeline": "The decline has been going on for weeks. The phone is about four years old and you bought it used.",
        },
    },
    "firmware": {
        "label": "Software / firmware",
        "cause": "Software / firmware failure after an update",
        "repairable": True,
        "clues": {
            "environmental": "Nothing unusual — no water, heat or cold.",
            "physical": "No drops, no damage.",
            "power": "When you plug it in, the logo flickers on for a second, then it dies, over and over.",
            "timeline": "It started right after an overnight system update three days ago. The phone is a year and a half old, bought new.",
        },
    },
}

## ===== Customer personalities ==========================================================
MOODS = ["ABUSIVE", "ANGRY", "IRRITATED", "NEUTRAL", "POSITIVE"]
MOOD_START_SATISFACTION = {"ABUSIVE": 1.0, "ANGRY": 3.0, "IRRITATED": 5.0, "NEUTRAL": 7.0, "POSITIVE": 9.0}
MOOD_EMOJI = {"ABUSIVE": "🤬", "ANGRY": "😠", "IRRITATED": "😒", "NEUTRAL": "😐", "POSITIVE": "🙂"}

PERSONALITIES = {
    "ABUSIVE": (
        "You are furious and openly hostile. Use short, sharp sentences. You may insult the store and its "
        "products (no slurs, no threats), interrupt, and demand a manager or a free replacement right now. "
        "You answer questions grudgingly with as little detail as possible."
    ),
    "ANGRY": (
        "You are angry and impatient. You complain about wasted time and are sarcastic. You answer questions, "
        "but tersely, sometimes with a jab. You may demand a discount or a new phone before anything is diagnosed."
    ),
    "IRRITATED": (
        "You are annoyed but cooperating. Some sighing and mild sarcasm. You answer questions with the relevant "
        "details when asked, though you still want this over with."
    ),
    "NEUTRAL": (
        "You are calm and matter-of-fact. You answer questions fully and are willing to hear the options."
    ),
    "POSITIVE": (
        "You feel relieved and appreciate the help. You are friendly, thank the employee, and are open to "
        "reasonable offers."
    ),
}

OPENERS = {
    "ABUSIVE": "Are you kidding me? This piece of junk you people sold me is DEAD. Fix it. Now.",
    "ANGRY": "My phone won't turn on, and I've already wasted half my day on this. I need it fixed.",
    "IRRITATED": "Hi. My phone won't turn on, and honestly I'm pretty annoyed about it.",
    "NEUTRAL": "Hello. I'm having issues with my phone — it won't turn on.",
    "POSITIVE": "Hi there! My phone stopped turning on — hoping you can help me out.",
}

## ===== SYSTEM Prompts ===============================================================
CUSTOMER_BASE = """You are role-playing a CUSTOMER at a mobile phone store in an employee training simulator.
Your phone will not turn on. The person talking to you is a store employee.

Hard rules:
- Stay in character as the customer at all times. Never mention training, ratings, scores, violations, or that you are an AI.
- You do NOT know what is wrong with the phone or whether it can be repaired. Never diagnose it yourself.
- Only share facts listed under WHAT YOU CAN SHARE. If asked about anything else, say you don't know or haven't noticed anything.
- Don't volunteer facts the employee hasn't asked about.
- Keep replies short: 1-4 sentences, like a real person at a counter.
- Never use slurs or threats of violence.
- Your tone must match YOUR CURRENT MOOD exactly, even if the employee is being very nice or very rude."""

CLASSIFIER_PROMPT = """You are a strict evaluator for a retail customer-service training simulator.
Classify ONLY the employee's LATEST message. Return a JSON object with exactly these keys:
- "tone": "polite", "neutral" or "rude". Rude = insults, profanity, condescension, or sarcasm aimed at the customer. Firm but polite is NOT rude. The customer's hostility never makes the employee rude.
- "empathy": true if the employee acknowledges the customer's frustration or apologizes for the inconvenience.
- "off_topic": true if the message is unrelated to the phone problem or the store visit.
- "refused_to_engage": true if the employee dismisses the customer, refuses to help, or tells them to leave.
- "accusatory": true if a question or statement assumes the customer is to blame (e.g. "Did you drop it in water?", "You probably left it in the sun").
- "question_categories": list of the diagnostic areas the message asks about, chosen from "environmental" (water, moisture, heat, sun, cold), "physical" (drops, cracks, bends, swelling, case), "power" (charging, cables, port, warning signs, restarts, battery), "timeline" (when it started, prior repairs, warranty, bought used).
- "deferred_discount": true if the employee politely explains a discount can only be discussed after diagnosis.
- "diagnosis": null, "repairable" or "not_repairable" — only if the employee states a conclusion about whether the phone can be fixed.
- "offers": list of discounts offered in this message, each {"kind": "parts" | "labor" | "new_phone" | "other", "percent": number, "trade_in": true/false}. Empty list if none."""

COACH_SYSTEM = """You coach a retail employee in a training simulator. The employee is helping a customer whose phone won't turn on.

Store policy:
- Diagnose first with neutral, non-accusatory questions: water/heat/cold exposure, drops and physical damage, charging behaviour, timeline/warranty.
- Not repairable: up to 30% off a new phone, only with trade-in of the broken phone.
- Repairable: up to 40% off parts only. Labor is never discounted.
- If the customer demands a discount early, politely explain diagnosis comes first.

Each message gives what the employee said, the customer's reply, and a SIM STATE line computed by the simulator.
Treat SIM STATE as ground truth for mood, violations and progress; never contradict it.
You do NOT know the real cause of the problem — never guess it for the employee.

Reply with:
1. **Tone check** — one or two sentences on the employee's last message.
2. **Suggested next moves** — 2-3 options, each with an example line to say and why it suits the customer's current mood.
Stay under 150 words."""

FEEDBACK_PROMPT = """You write the end-of-session debrief for an employee in a customer-service training simulator.
The final rating has ALREADY been decided by the simulator. Do not change it or mention any other number.
Write 3-5 short bullet points of specific, constructive feedback that reference real moments from the transcript.
Start with what went well (if anything), then what to improve."""

STATUS_LABEL = {
    "active": "In progress",
    "resolved": "Resolved",
    "walked_out": "Customer walked out",
    "stalled": "Stalled",
    "ended_unresolved": "Ended without resolution",
}


## ===== Session state =============================================================
# Just one large object that keeps tracks of things
def new_session(start_mood="ANGRY", scenario_choice="Random"):
    scenario_id = random.choice(list(SCENARIOS)) if scenario_choice in (None, "Random") else scenario_choice
    return {
        "scenario_id": scenario_id,
        "mood": start_mood,
        "satisfaction": MOOD_START_SATISFACTION[start_mood],
        "violations": 0,
        "asked": [],
        "accusatory_count": 0,
        "diagnosed": False,
        "valid_offer": None,
        "accepted": None,
        "status": "active",
        "turns": 0,
        "final_rating": None,
        "log": [],
        "last_turn": "",
        "customer_history": [{"role": "assistant", "content": OPENERS[start_mood]}],
        "coach_history": [],
    }


def mood_for(satisfaction):
    for threshold, mood in zip([2, 4, 6, 8], MOODS):
        if satisfaction < threshold:
            return mood
    return "POSITIVE"


def step_mood(current, satisfaction):
    """Move at most one mood level per turn so changes feel gradual and predictable."""
    i, j = MOODS.index(current), MOODS.index(mood_for(satisfaction))
    if j > i:
        return MOODS[i + 1]
    if j < i:
        return MOODS[i - 1]
    return current


def describe_offer(o):
    if o["kind"] == "parts":
        return f"{o['percent']:g}% off parts (labor at full price)"
    if o["kind"] == "new_phone":
        return f"{o['percent']:g}% off a new phone with trade-in"
    return f"{o['percent']:g}% off ({o['kind']})"


## ===== Classify the employee message =============================================================
# A separate LLM API call will be made to classify whatever the employee sends,
# that is structured in the JSON format with all these key/value pairs
# We just need to extract that and store as object in code
def normalize_classification(classifer_json):
    offers = []
    for offer in classifer_json.get("offers") or []:
        if not isinstance(offer, dict):
            continue
        try:
            pct = float(offer.get("percent") or 0)
        except (TypeError, ValueError):
            pct = 0.0
        kind = str(offer.get("kind", "other")).lower()
        offers.append({
            "kind": kind if kind in ("parts", "labor", "new_phone") else "other",
            "percent": pct,
            "trade_in": bool(offer.get("trade_in")),
        })
    diagnosis = classifer_json.get("diagnosis")
    return {
        "tone": classifer_json.get("tone") if classifer_json.get("tone") in ("polite", "neutral", "rude") else "neutral",
        "empathy": bool(classifer_json.get("empathy")),
        "off_topic": bool(classifer_json.get("off_topic")),
        "refused_to_engage": bool(classifer_json.get("refused_to_engage")),
        "accusatory": bool(classifer_json.get("accusatory")),
        "question_categories": [c for c in (classifer_json.get("question_categories") or []) if c in CATEGORIES],
        "deferred_discount": bool(classifer_json.get("deferred_discount")),
        "diagnosis": diagnosis if diagnosis in ("repairable", "not_repairable") else None,
        "offers": offers,
    }


def classify_employee_message(session, employee_message):
    recent = session["customer_history"][-7:-1]
    transcript = "\n".join(
        f"{'Employee' if m['role'] == 'user' else 'Customer'}: {m['content']}" for m in recent
    )
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": CLASSIFIER_PROMPT},
                {"role": "user", "content": f"Recent conversation:\n{transcript}\n\nLATEST employee message:\n{employee_message}"},
            ],
        )
        return normalize_classification(json.loads(resp.choices[0].message.content))
    except Exception as e:
        print(f"[classifier error] {e}")
        return None


## ===== Applies the business rules =============================================================
def check_offer(o, scenario, s):
    pct = o["percent"]
    if pct <= 0:
        return "ignore", ""
    if o["kind"] == "labor":
        return "invalid", f"offered {pct:g}% off labor (labor is never discounted)"
    if o["kind"] == "other":
        return "invalid", f"offered an unapproved {pct:g}% discount"
    if scenario["repairable"]:
        if o["kind"] != "parts":
            return "invalid", "offered a new-phone discount on a phone that can be repaired"
        if pct > MAX_PARTS_DISCOUNT:
            return "invalid", f"offered {pct:g}% off parts (max is {MAX_PARTS_DISCOUNT}%)"
    else:
        if o["kind"] != "new_phone":
            return "invalid", "offered a repair discount on a phone that can't be repaired"
        if pct > MAX_NEW_PHONE_DISCOUNT:
            return "invalid", f"offered {pct:g}% off a new phone (max is {MAX_NEW_PHONE_DISCOUNT}%)"
        if not o["trade_in"]:
            return "invalid", "offered a new-phone discount without requiring a trade-in"
    if len(s["asked"]) < MIN_CATEGORIES_FOR_DIAGNOSIS:
        return "premature", ""
    return "valid", ""


def empty_turn():
    return {"violation_reasons": [], "accusatory": False, "premature_offer": False, "notes": []}


def apply_rules(session, employee_message_classification):
    scenario = SCENARIOS[session["scenario_id"]]
    turn_info = empty_turn()
    delta = 0.0

    if employee_message_classification["tone"] == "polite":
        delta += DELTAS["polite"]
    if employee_message_classification["empathy"]:
        delta += DELTAS["empathy"]
        turn_info["notes"].append("acknowledged the customer's frustration")
    if employee_message_classification["accusatory"]:
        delta += DELTAS["accusatory"]
        session["accusatory_count"] += 1
        turn_info["accusatory"] = True
        turn_info["notes"].append("accusatory phrasing")
    if employee_message_classification["deferred_discount"]:
        delta += DELTAS["deferred_discount"]
        turn_info["notes"].append("politely explained diagnosis comes first")

    new_areas = [cat for cat in employee_message_classification["question_categories"] if cat not in session["asked"]]
    if new_areas:
        session["asked"].extend(new_areas)
        delta += DELTAS["new_category"] * len(new_areas)
        turn_info["notes"].append("asked about " + ", ".join(new_areas))

    if employee_message_classification["tone"] == "rude":
        turn_info["violation_reasons"].append("rude or disrespectful language")
    if employee_message_classification["off_topic"]:
        turn_info["violation_reasons"].append("went off-topic")
    if employee_message_classification["refused_to_engage"]:
        turn_info["violation_reasons"].append("refused to engage")

    if employee_message_classification["diagnosis"]:
        correct = (employee_message_classification["diagnosis"] == "repairable") == scenario["repairable"]
        if not correct:
            delta += DELTAS["wrong_diagnosis"]
            turn_info["notes"].append("incorrect diagnosis")
        elif len(session["asked"]) < MIN_CATEGORIES_FOR_DIAGNOSIS:
            turn_info["notes"].append("jumped to a diagnosis without enough questions")
        elif not session["diagnosed"]:
            session["diagnosed"] = True
            delta += DELTAS["correct_diagnosis"]
            turn_info["notes"].append("correct diagnosis")

    valid_offer = None
    for o in employee_message_classification["offers"]:
        status, reason = check_offer(o, scenario, session)
        if status == "invalid":
            turn_info["violation_reasons"].append(reason)
        elif status == "premature":
            turn_info["premature_offer"] = True
            delta += DELTAS["premature_offer"]
            turn_info["notes"].append("made an offer before diagnosing")
        elif status == "valid":
            valid_offer = o

    if turn_info["violation_reasons"]:
        session["violations"] += 1
        delta += DELTAS["violation"]
    elif valid_offer:
        session["valid_offer"] = valid_offer
        session["diagnosed"] = True
        delta += DELTAS["valid_offer"]
        turn_info["notes"].append("valid offer: " + describe_offer(valid_offer))

    session["satisfaction"] = max(0.0, min(10.0, session["satisfaction"] + delta))
    session["mood"] = step_mood(session["mood"], session["satisfaction"])

    # End conditions
    if session["violations"] >= MAX_VIOLATIONS:
        session["status"] = "walked_out"
    elif session["valid_offer"]:
        session["accepted"] = session["satisfaction"] >= ACCEPT_THRESHOLD
        session["status"] = "resolved"
    elif session["turns"] >= MAX_TURNS:
        session["status"] = "stalled"

    parts = turn_info["notes"] + [f"VIOLATION: {r}" for r in t["violation_reasons"]]
    session["last_turn"] = "; ".join(parts) or "no notable events"
    session["log"].append(f"Turn {session['turns']}: {session['last_turn']} (satisfaction {session['satisfaction']:.1f}, mood {session['mood']})")
    return turn_info


## === Guide Customer LLM prompt ===
def build_directive(session, turn_info):
    if session["status"] == "walked_out":
        return "The employee has crossed the line one too many times. Walk out of the store now in 1-2 sentences. Do not ask anything else."
    if session["status"] == "resolved":
        offer = describe_offer(session["valid_offer"])
        if session["accepted"]:
            return f"The employee offered: {offer}. Accept it in character and wrap up the conversation."
        return f"The employee offered: {offer}. You're still unhappy, so turn it down in character and leave."
    if session["status"] == "stalled":
        return "You've been here far too long without a solution. Say you're done waiting and leave."
    
    lines = []
    if turn_info["violation_reasons"]:
        lines.append(f"The employee just {'; '.join(turn_info['violation_reasons'])}. React to that in character — you're offended.")
    if turn_info["accusatory"]:
        lines.append("The employee's question sounded like it blamed you. Get a bit defensive, but answer if it's relevant.")
    if turn_info["premature_offer"]:
        lines.append("The employee offered a deal before figuring out what's wrong. Be skeptical that they can offer anything without checking the phone.")
    return "\n".join(lines)


def build_customer_prompt(session, directive):
    scenario = SCENARIOS[session["scenario_id"]]
    facts = ["- Your phone won't turn on."] + [f"- {scenario['clues'][cat]}" for cat in session["asked"]]
    prompt = (
        f"{CUSTOMER_BASE}\n\n## YOUR CURRENT MOOD: {session['mood']}\n{PERSONALITIES[session['mood']]}"
        f"\n\n## WHAT YOU CAN SHARE\n" + "\n".join(facts)
    )
    if directive:
        prompt += f"\n\n## WHAT TO DO THIS TURN\n{directive}"
    return prompt


def stream_completion(system, history, temperature=0.8):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}] + history,
        temperature=temperature,
        stream=True,
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content


## === Scoring & report ===
def compute_rating(session):
    if session["status"] == "walked_out":
        return 1
    if session["status"] != "resolved":
        return 2
    rating = 8
    if session["violations"] == 0:
        rating += 1
    if len(session["asked"]) >= 3 and session["accusatory_count"] == 0:
        rating += 1
    return rating


def build_report(session):
    rating = compute_rating(session)
    session["final_rating"] = rating
    if session["status"] == "resolved":
        outcome = "Customer accepted the offer." if session["accepted"] else "Customer refused a fair, policy-compliant offer — you followed policy correctly."
    else:
        outcome = {
            "walked_out": "Customer walked out after 3 violations.",
            "stalled": "The conversation stalled without a resolution.",
            "ended_unresolved": "Session ended without a diagnosis and a valid offer.",
        }[session["status"]]

    transcript = "\n".join(
        f"{'Employee' if m['role'] == 'user' else 'Customer'}: {m['content']}" for m in session["customer_history"]
    )
    summary = (
        f"Final rating: {rating}/10\nOutcome: {outcome}\nViolations: {session['violations']}\n"
        f"Diagnostic areas covered: {', '.join(session['asked']) or 'none'}\n"
        f"Accusatory questions: {session['accusatory_count']}\n\nTurn log:\n" + "\n".join(session["log"])
        + f"\n\nTranscript:\n{transcript}"
    )
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            temperature=0.4,
            messages=[{"role": "system", "content": FEEDBACK_PROMPT}, {"role": "user", "content": summary}],
        )
        feedback = resp.choices[0].message.content
    except Exception as e:
        print(f"[feedback error] {e}")
        feedback = "\n".join(f"- {line}" for line in s["log"][-5:]) or "- No turns were scored."
    return f"### 📋 Session report\n**Final rating: {rating}/10** — {outcome}\n\n{feedback}"


## === Sidebar renderers ===
def render_stats(session):
    if not session:
        return "_No session yet._"
    filled = int(round(session["satisfaction"]))
    bar = "█" * filled + "░" * (10 - filled)
    lines = [
        f"**Customer mood:** {MOOD_EMOJI[session['mood']]} {session['mood'].title()}",
        f"**Satisfaction:** `{bar}` {session['satisfaction']:.1f}/10",
        f"**Violations:** {session['violations']} / {MAX_VIOLATIONS}",
        f"**Diagnostic areas covered:** {', '.join(session['asked']) or 'none yet'}",
        f"**Valid offer:** {describe_offer(session['valid_offer']) if session['valid_offer'] else 'none yet'}",
        f"**Status:** {STATUS_LABEL[session['status']]} · turn {session['turns']}/{MAX_TURNS}",
    ]
    if session["final_rating"] is not None:
        lines.append(f"\n### Final rating: {session['final_rating']}/10")
    if session["last_turn"]:
        lines.append(f"\n_Last turn:_ {session['last_turn']}")
    return "  \n".join(lines)


def render_instructor(session):
    if not session:
        return ""
    scenario = SCENARIOS[session["scenario_id"]]
    allowed = (
        f"Up to {MAX_PARTS_DISCOUNT}% off parts, no labor discount"
        if scenario["repairable"]
        else f"Up to {MAX_NEW_PHONE_DISCOUNT}% off a new phone, trade-in required"
    )
    return (
        f"**Scenario:** {scenario['label']}  \n**Cause:** {scenario['cause']}  \n"
        f"**Repairable:** {'Yes' if scenario['repairable'] else 'No'}  \n**Allowed offer:** {allowed}"
    )


def sim_state_line(session, turn_info):
    flags = turn_info["notes"] + [f"violation: {r}" for r in turn_info["violation_reasons"]]
    return (
        f"mood={session['mood']}, satisfaction={session['satisfaction']:.1f}/10, violations={session['violations']}/{MAX_VIOLATIONS}, "
        f"areas_covered=[{', '.join(session['asked'])}], status={session['status']}, this_turn=[{'; '.join(flags) or 'nothing notable'}]"
    )


## === Gradio handlers ===
def start_session(start_mood, scenario_choice):
    session = new_session(start_mood, scenario_choice)
    customer_ui = [{"role": "assistant", "content": session["customer_history"][0]["content"]}]
    coach_ui = [{"role": "assistant", "content": "I'll coach you after each exchange with the customer. Start by greeting them."}]
    return "", customer_ui, coach_ui, session, render_stats(session), render_instructor(session)

# ==================================================================================
# ==================================================================================
# ============================= (Main Chat Function) ===============================
# ==================================================================================
# ==================================================================================
def chat_turn(user_message, session, customer_ui, coach_ui):
    customer_ui = list(customer_ui or [])
    coach_ui = list(coach_ui or [])

    def out():
        return "", customer_ui, coach_ui, session, render_stats(session), render_instructor(session)

    # Base Cases
    if session is None or not user_message or not user_message.strip():
        yield user_message or "", customer_ui, coach_ui, session, render_stats(session), render_instructor(session)
        return
    if session["status"] != "active":
        gr.Warning("This session is over. Click 'New customer' to start another.")
        yield out()
        return

    # Start handling the msgs
    msg = user_message.strip()
    session["turns"] += 1
    session["customer_history"].append({"role": "user", "content": msg})
    customer_ui.append({"role": "user", "content": msg})
    yield out()

    ## Classify employee message using making a sep API call
    employee_message_classification = classify_employee_message(session, msg)
    if employee_message_classification is None:
        chat_turn_info = empty_turn()
        session["last_turn"] = "Evaluator unavailable — this turn wasn't scored."
        if session["turns"] >= MAX_TURNS:
            session["status"] = "stalled"
    else:
        chat_turn_info = apply_rules(session, employee_message_classification)

    system = build_customer_prompt(session, build_directive(session, chat_turn_info))
    customer_ui.append({"role": "assistant", "content": ""})
    reply = ""
    try:
        for token in stream_completion(system, session["customer_history"]):
            reply += token
            customer_ui[-1]["content"] = reply
            yield out()
    except Exception as e:
        print(f"[customer error] {e}")
        customer_ui[-1]["content"] = "Couldn't reach the customer model. Please try sending again."
        yield out()
    if reply:
        session["customer_history"].append({"role": "assistant", "content": reply})

    if session["status"] != "active":
        customer_ui.append({"role": "assistant", "content": build_report(session)})
        yield out()

    coach_input = f"Employee said: {msg}\nCustomer replied: {reply}\nSIM STATE: {sim_state_line(session, chat_turn_info)}"
    session["coach_history"] = (session["coach_history"] + [{"role": "user", "content": coach_input}])[-20:]
    coach_ui.append({"role": "user", "content": f"**Employee:** {msg}\n\n**Customer:** {reply}"})
    coach_ui.append({"role": "assistant", "content": ""})
    coach_reply = ""
    try:
        for token in stream_completion(COACH_SYSTEM, session["coach_history"], temperature=0.5):
            coach_reply += token
            coach_ui[-1]["content"] = coach_reply
            yield out()
    except Exception as e:
        print(f"[coach error] {e}")
        coach_ui[-1]["content"] = "Coach unavailable for this turn."
    if coach_reply:
        session["coach_history"].append({"role": "assistant", "content": coach_reply})
    yield out()


def end_session(session, customer_ui, coach_ui):
    customer_ui = list(customer_ui or [])
    if session and session["status"] == "active":
        session["status"] = "ended_unresolved"
        customer_ui.append({"role": "assistant", "content": build_report(session)})
    return "", customer_ui, coach_ui, session, render_stats(session), render_instructor(session)


## ===== UI ==========================================================
gr.close_all()

custom_css = """
#custom-input textarea {
    border: 2px solid #ff4b4b4d !important;
    border-radius: 8px !important;
}

#custom-input button {
    width: 8rem !important;
    height: 2.5rem !important;
}

#new_customer_button {
    width: fit !important;
    height: 2.5rem !important;
}
"""

scenario_choices = [("Random", "Random")] + [(v["label"], k) for k, v in SCENARIOS.items()]

with gr.Blocks(title="AI Employee Training", css=custom_css) as demo:
    session = gr.State(None)

    # Sidebar with Coach and Session Stats
    with gr.Sidebar(position="left", open=True):
        with gr.Tabs():
            with gr.Tab("Coach"):
                coach_box = gr.Chatbot(label="Trainer / Analyst Coach", height=560)
            with gr.Tab("Session"):
                stats_md = gr.Markdown()
                with gr.Accordion("Instructor view", open=False):
                    instructor_md = gr.Markdown()
 
    gr.Markdown("# 🤖 AI Employee Training System")
    with gr.Row():
        mood_dropdown = gr.Dropdown(MOODS, value="ANGRY", label="Customer Starting mood")
        scenario_dropdown = gr.Dropdown(scenario_choices, value="Random", label="Scenario")
    
    new_customer_btn = gr.Button("Start new conversation with new customer", variant="secondary", elem_id="new_customer_button")

    customer_box = gr.Chatbot(label="Customer", height=480)
    with gr.Row():
        user_input = gr.Textbox(
            label="Your response to the customer",
            placeholder="Hello! How can I help you today?",
            elem_id="custom-input",
            submit_btn="Send",
            scale=4,
        )
    end_btn = gr.Button("End session & get rating", variant="stop")

    outputs = [user_input, customer_box, coach_box, session, stats_md, instructor_md]
    chat_inputs = [user_input, session, customer_box, coach_box]

    user_input.submit(chat_turn, chat_inputs, outputs)
    new_customer_btn.click(start_session, [mood_dropdown, scenario_dropdown], outputs)
    end_btn.click(end_session, [session, customer_box, coach_box], outputs)
    demo.load(start_session, [mood_dropdown, scenario_dropdown], outputs)

demo.launch()

/var/folders/4w/_8k7d7n53fv625qygjhxdqrm0000gn/T/ipykernel_1806/2362918547.py:659: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="AI Employee Training", css=custom_css) as demo:


Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
